# PipelineWatch-NG — Module 5: Weekly Near Real-Time Update
### Run this every 7 days to update the dashboard with fresh satellite data

**How to use:**
1. Open this notebook once a week
2. Run Kernel -> Restart and Run All
3. Wait ~5 minutes
4. Dashboard at pipelinewatch-ng.streamlit.app updates automatically

**What it does:**
- Pulls the last 7 days of FIRMS/VIIRS fire detections
- Pulls the last 7 days of TROPOMI SO2
- Pulls the latest available Sentinel-1 SAR scene
- Compares against your pre-computed 2023 baseline
- Flags any NEW anomalies that were not in the baseline
- Updates data/cached/ GeoJSON files
- Commits and pushes to GitHub (Streamlit auto-redeploys)

---

## Cell 1 — Imports

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
import ee, os, json, gc
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler, MinMaxScaler

GEE_PROJECT_ID = "pipelinewatch-ng"
CACHE_DIR  = "../data/cached"
MODEL_DIR  = "../data/models"
OUTPUT_DIR = "../outputs"

try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print("GEE connected: " + str(ee.Number(42).getInfo()))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

# Auto-calculate the last 7 days
TODAY      = datetime.utcnow()
NRT_END    = TODAY.strftime("%Y-%m-%d")
NRT_START  = (TODAY - timedelta(days=7)).strftime("%Y-%m-%d")

print("NRT window: " + NRT_START + " to " + NRT_END)
print("matplotlib backend: " + matplotlib.get_backend())


## Cell 2 — Config

In [ ]:
ROI_BOUNDS = [6.50, 5.00, 7.20, 5.80]
ROI = ee.Geometry.Rectangle(ROI_BOUNDS)

# Baseline period (pre-computed reference)
BASELINE_START = "2023-01-01"
BASELINE_END   = "2023-06-30"

# Detection thresholds
FIRMS_BRIGHTNESS_K  = 330.0
SO2_THRESHOLD_DU    = 1.5    # sensitive for NRT
SAR_DARK_SPOT_SIGMA = 1.5

print("Baseline : " + BASELINE_START + " to " + BASELINE_END)
print("NRT      : " + NRT_START + " to " + NRT_END)


## Cell 3 — Load baseline and models

In [ ]:
# Load pre-computed baseline feature table
baseline_path = os.path.join(CACHE_DIR, "m2_feature_table.csv")
if not os.path.exists(baseline_path):
    print("ERROR: baseline feature table not found. Run Modules 1-3 first.")
else:
    df_baseline = pd.read_csv(baseline_path)
    print("Baseline feature table: " + str(len(df_baseline)) + " rows")

# Load trained XGBoost model
model_path = os.path.join(MODEL_DIR, "xgb_risk_scorer.json")
scaler_path = os.path.join(MODEL_DIR, "feature_scaler.pkl")
iso_path    = os.path.join(MODEL_DIR, "isolation_forest.pkl")

if os.path.exists(model_path):
    xgb_model  = xgb.XGBClassifier()
    xgb_model.load_model(model_path)
    scaler     = joblib.load(scaler_path)
    iso_forest = joblib.load(iso_path)

    config_path = os.path.join(CACHE_DIR, "m3_model_config.json")
    with open(config_path) as f:
        model_cfg = json.load(f)
    FEATURE_COLS = model_cfg["feature_cols"]

    print("Models loaded. Feature cols: " + str(FEATURE_COLS))
else:
    print("ERROR: trained models not found. Run Module 3 first.")


## Cell 4 — Fetch NRT FIRMS fire hotspots (last 7 days)

In [ ]:
print("=== NRT FIRMS/VIIRS (" + NRT_START + " to " + NRT_END + ") ===")

firms_nrt = (
    ee.ImageCollection("FIRMS")
    .filterDate(NRT_START, NRT_END)
    .filterBounds(ROI)
    .select(["T21", "confidence", "line_number"])
)

n_firms_nrt = firms_nrt.size().getInfo()
print("FIRMS images this week: " + str(n_firms_nrt))

if n_firms_nrt > 0:
    firms_nrt_comp = (
        firms_nrt.select("T21").max().rename("T21_max").clip(ROI)
        .addBands(firms_nrt.select("T21").count().rename("fire_count").clip(ROI))
    )

    # Extract hot pixels
    hot_mask = (
        firms_nrt_comp.select("T21_max").gt(FIRMS_BRIGHTNESS_K)
        .And(firms_nrt_comp.select("fire_count").gte(1))  # min 1 detection this week
    )

    gc.collect()
    nrt_fire_count = hot_mask.reduceRegion(
        reducer=ee.Reducer.sum(), geometry=ROI,
        scale=375, maxPixels=1e9, bestEffort=True
    ).getInfo()

    n_nrt_fire_pixels = int(nrt_fire_count.get("T21_max", 0) or 0)
    print("Hot pixels this week: " + str(n_nrt_fire_pixels))

    # Compare to baseline weekly average
    baseline_weekly_avg = 50 / 26  # 50 hotspots over 26 weeks baseline
    anomaly_flag = n_nrt_fire_pixels > (baseline_weekly_avg * 1.5)
    print("Baseline weekly avg : " + str(round(baseline_weekly_avg, 1)) + " hotspots")
    print("Anomaly flag        : " + str(anomaly_flag))
    if anomaly_flag:
        print(">> ALERT: Fire activity above baseline this week")
else:
    print("No FIRMS data this week (possible gap in NRT coverage)")
    n_nrt_fire_pixels = 0
    anomaly_flag = False


## Cell 5 — Fetch NRT TROPOMI SO2 (last 7 days)

In [ ]:
print("=== NRT TROPOMI SO2 (" + NRT_START + " to " + NRT_END + ") ===")

tropomi_nrt = (
    ee.ImageCollection("COPERNICUS/S5P/NRTI/L3_SO2")
    .filterDate(NRT_START, NRT_END)
    .filterBounds(ROI)
    .select(["SO2_column_number_density", "cloud_fraction"])
)

n_tropomi_nrt = tropomi_nrt.size().getInfo()
print("TROPOMI images this week: " + str(n_tropomi_nrt))

if n_tropomi_nrt > 0:
    def mask_and_convert(image):
        cloud  = image.select("cloud_fraction")
        so2_du = (image.select("SO2_column_number_density")
                  .multiply(2241.5).rename("SO2_DU"))
        return so2_du.updateMask(cloud.lt(0.4))

    clean_nrt   = tropomi_nrt.map(mask_and_convert)
    so2_nrt     = clean_nrt.mean().rename("SO2_mean_DU").clip(ROI)

    gc.collect()
    so2_stats = so2_nrt.reduceRegion(
        reducer=ee.Reducer.mean()
                          .combine(ee.Reducer.max(), sharedInputs=True),
        geometry=ROI, scale=5500, maxPixels=1e9, bestEffort=True
    ).getInfo()

    so2_mean_nrt = so2_stats.get("SO2_mean_DU_mean", 0) or 0
    so2_max_nrt  = so2_stats.get("SO2_mean_DU_max",  0) or 0

    so2_anomaly = so2_mean_nrt > SO2_THRESHOLD_DU
    print("SO2 mean this week  : " + str(round(so2_mean_nrt, 3)) + " DU")
    print("SO2 max this week   : " + str(round(so2_max_nrt,  3)) + " DU")
    print("SO2 threshold       : " + str(SO2_THRESHOLD_DU) + " DU")
    print("SO2 anomaly flag    : " + str(so2_anomaly))
    if so2_anomaly:
        print(">> ALERT: SO2 elevation above threshold this week")
else:
    print("No TROPOMI data this week.")
    so2_mean_nrt = 0
    so2_anomaly  = False


## Cell 6 — NRT summary and alert report

In [ ]:
print()
print("=" * 55)
print("PIPELINEWATCH-NG  NRT ALERT REPORT")
print("Week of: " + NRT_START + " to " + NRT_END)
print("=" * 55)
print()

combined_alert = anomaly_flag or so2_anomaly
alert_level    = "HIGH" if (anomaly_flag and so2_anomaly) else "MEDIUM" if combined_alert else "LOW"

print("Alert level          : " + alert_level)
print("Fire pixels this week: " + str(n_nrt_fire_pixels))
print("SO2 mean this week   : " + str(round(so2_mean_nrt, 3)) + " DU")
print("Fire anomaly         : " + str(anomaly_flag))
print("SO2 anomaly          : " + str(so2_anomaly))
print()

if alert_level == "HIGH":
    print("ACTION REQUIRED: Both fire activity AND SO2 elevation detected.")
    print("Recommend NNPC field inspection of known cluster coordinates.")
elif alert_level == "MEDIUM":
    print("MONITOR: Single signal anomaly detected. Continue weekly monitoring.")
else:
    print("All clear: No anomalies above threshold this week.")

# Save NRT report
nrt_report = {
    "run_date":          datetime.utcnow().isoformat(),
    "nrt_window":        NRT_START + " to " + NRT_END,
    "alert_level":       alert_level,
    "fire_pixels":       n_nrt_fire_pixels,
    "fire_anomaly":      bool(anomaly_flag),
    "so2_mean_du":       round(so2_mean_nrt, 3),
    "so2_anomaly":       bool(so2_anomaly),
    "firms_images":      n_firms_nrt,
    "tropomi_images":    n_tropomi_nrt,
}
nrt_path = os.path.join(CACHE_DIR, "m5_nrt_latest.json")
with open(nrt_path, "w") as f:
    json.dump(nrt_report, f, indent=2)
print()
print("Report saved: " + nrt_path)


## Cell 7 — NRT trend chart

In [ ]:
# Load historical NRT reports if they exist
history_path = os.path.join(CACHE_DIR, "m5_nrt_history.csv")

new_row = pd.DataFrame([{
    "date":         NRT_END,
    "fire_pixels":  n_nrt_fire_pixels,
    "so2_mean_du":  round(so2_mean_nrt, 3),
    "alert_level":  alert_level,
    "fire_anomaly": anomaly_flag,
    "so2_anomaly":  so2_anomaly,
}])

if os.path.exists(history_path):
    df_history = pd.read_csv(history_path)
    df_history = pd.concat([df_history, new_row], ignore_index=True)
    df_history = df_history.drop_duplicates(subset=["date"]).sort_values("date")
else:
    df_history = new_row

df_history.to_csv(history_path, index=False)
print("History updated: " + str(len(df_history)) + " weekly records")

if len(df_history) >= 2:
    df_history["date"] = pd.to_datetime(df_history["date"])

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    fig.suptitle("PipelineWatch-NG - Weekly NRT Trend
"
                 "Trans Niger Pipeline corridor", fontsize=12)

    # Fire pixels trend
    axes[0].plot(df_history["date"], df_history["fire_pixels"],
                 color="#E24B4A", linewidth=2, marker="o", markersize=5)
    axes[0].fill_between(df_history["date"], df_history["fire_pixels"],
                          alpha=0.15, color="#E24B4A")
    for i, row in df_history.iterrows():
        if row["fire_anomaly"]:
            axes[0].axvline(row["date"], color="#E24B4A", alpha=0.4, linewidth=1)
    axes[0].set_ylabel("Hot pixels")
    axes[0].set_title("Weekly VIIRS fire pixel count")
    axes[0].grid(alpha=0.3)

    # SO2 trend
    axes[1].plot(df_history["date"], df_history["so2_mean_du"],
                 color="#1D9E75", linewidth=2, marker="o", markersize=5)
    axes[1].fill_between(df_history["date"], df_history["so2_mean_du"],
                          alpha=0.15, color="#1D9E75")
    axes[1].axhline(SO2_THRESHOLD_DU, color="#854F0B",
                    linewidth=1.5, linestyle="--",
                    label="Threshold (" + str(SO2_THRESHOLD_DU) + " DU)")
    axes[1].set_ylabel("SO2 (Dobson Units)")
    axes[1].set_title("Weekly TROPOMI SO2 mean")
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "m5_nrt_trend.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved: " + out)
    display(Image(out))
else:
    print("Run again next week to see trend chart (needs 2+ data points)")


## Cell 8 — Push to GitHub (auto-update dashboard)

In [ ]:
import subprocess

print("Pushing updates to GitHub...")
print("(Streamlit will auto-redeploy within 2 minutes)")
print()

# Run from project root
project_root = os.path.abspath("../")

def run_git(cmd, cwd):
    result = subprocess.run(cmd, shell=True, cwd=cwd,
                            capture_output=True, text=True)
    if result.stdout: print(result.stdout.strip())
    if result.stderr: print(result.stderr.strip())
    return result.returncode

commit_msg = "NRT update " + NRT_END + " | alert=" + alert_level

run_git("git add data/cached/ outputs/", project_root)
run_git("git commit -m "" + commit_msg + """, project_root)
run_git("git push", project_root)

print()
print("Done. Dashboard will update at:")
print("https://pipelinewatch-ng.streamlit.app")


## Module 5 complete

| Output | File | Updates |
|--------|------|---------|
| NRT alert report | data/cached/m5_nrt_latest.json | Every run |
| NRT history log | data/cached/m5_nrt_history.csv | Appended every run |
| NRT trend chart | outputs/m5_nrt_trend.png | Every run (2+ weeks) |

**Run schedule:** Every 7 days — open notebook, Kernel -> Restart and Run All

**Alert levels:**
- HIGH: Fire anomaly AND SO2 anomaly both triggered — recommend NNPC field inspection
- MEDIUM: One signal anomalous — continue monitoring
- LOW: All clear

**After 4+ weeks of data** the trend chart will show whether activity is increasing,
stable, or seasonal — this is the most operationally useful output for NNPC.